# Merge Categories

Notebook finale usato per mergiare le varie categorie filtrando per metodo e versione

## Import

In [ ]:
from dl_client import DatalakeClient
from data_model.MergerTools import *
import pandas as pd
from itertools import product

client = DatalakeClient()
mergeTools = MergerTools()

## Download single merged files

In [ ]:
file_codes = ['VOLMERGE', 'SCALEMERGE'] #['VOLMERGE', 'SCALEMERGE', 'CSFMERGE', 'PLMERGE', 'PETMERGE', 'COFMERGE']
search = client.query_files(
    query={'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

In [ ]:
dfs = {}
for file_name, df_raw in zip_files.items():
    df = df_raw.copy()
    df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'])
    if 'FSVERSION' in df.columns:
        df['FSVERSION'] = df['FSVERSION'].astype(str)
    dfs[file_name] = df

In [ ]:
keys = list(dfs.keys())
print(keys)

## Merge single step

### Settings

- **BUFFER_DAYS**: finestra temporale (in giorni) per considerare due visite "vicine"
- **FILTER_COLUMNS**: mappa ogni tipo di dataset alla colonna che indica versione/metodo
- **VOLUME_COLS**: colonne volumetriche usate per valutare la qualità dei dati

In [ ]:
# Configurazione
BUFFER_DAYS = 80
RID_COL = 'RID'
DATE_COL = 'EXAMDATE'

# Filtri per tipo dataset
FILTER_COLUMNS = {
    'VOLUMES': 'FSVERSION',
    'SCALE': None,
    'CSF': 'METHOD_CSF',
    'PLASMA': 'METHOD_PLASMA',
    'PET': 'METHOD_PET',
    'COFACTOR': None
}

# Colonne volumetriche per valutare qualità
VOLUME_COLS = [
    'Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV',
    'Fusiform%ICV', 'MidTemp%ICV', 'ICV%ICV'
]

## Functions

### 2. Funzioni di Identificazione e Filtraggio

Funzioni per:
- Identificare il tipo di dataset dal nome del file
- Ottenere la colonna filtro corrispondente
- Generare tutte le combinazioni di filtri tra due dataset
- Applicare un filtro a un dataframe

In [ ]:
def identify_dataset_type(df_name):
    """
    Identifica il tipo di dataset dal nome del file.
    
    Cerca nel nome del file uno dei codici definiti in FILTER_COLUMNS.
    Es: "VOLUMES_merged.csv" -> contiene "VOLUMES"? No -> "VOLUMES" in upper? 
        Dipende dal naming, cerca match parziale.
    
    Args:
        df_name: nome del file (es. "VOLUMES_merged.csv")
    
    Returns:
        Codice del tipo dataset (es. "VOLUMES") o None se non riconosciuto
    """
    for code in FILTER_COLUMNS.keys():
        if code in df_name.upper():
            return code
    return None


def get_filter_column(df_name):
    """
    Restituisce la colonna da usare come filtro per il dataset.
    
    Args:
        df_name: nome del file
    
    Returns:
        Nome colonna filtro (es. "FSVERSION") o None se dataset senza filtro
    """
    ds_type = identify_dataset_type(df_name)
    if ds_type:
        return FILTER_COLUMNS[ds_type]
    return None


def get_filter_combinations(df1, df2, df1_name, df2_name):
    """
    Genera tutte le combinazioni di filtri tra due dataset.
    
    Se df1 ha FSVERSION con valori [5.1, 6.0, 7.0] e df2 non ha filtro,
    genera 3 combinazioni: (5.1, None), (6.0, None), (7.0, None).
    
    Se entrambi hanno filtri, genera il prodotto cartesiano.
    
    Args:
        df1, df2: dataframe da mergiare
        df1_name, df2_name: nomi dei file
    
    Returns:
        Lista di dizionari, ognuno con:
        - df1_filter_col: colonna filtro df1
        - df1_filter_val: valore filtro df1
        - df2_filter_col: colonna filtro df2
        - df2_filter_val: valore filtro df2
    """
    # Ottieni colonne filtro
    col1 = get_filter_column(df1_name)
    col2 = get_filter_column(df2_name)
    
    # Ottieni valori unici (o [None] se nessun filtro)
    vals1 = [None] if col1 is None else df1[col1].dropna().unique().tolist()
    vals2 = [None] if col2 is None else df2[col2].dropna().unique().tolist()
    
    # Genera prodotto cartesiano
    combinations = []
    for v1, v2 in product(vals1, vals2):
        combinations.append({
            'df1_filter_col': col1,
            'df1_filter_val': v1,
            'df2_filter_col': col2,
            'df2_filter_val': v2
        })
    
    return combinations


def apply_filter(df, filter_col, filter_val):
    """
    Applica un filtro al dataframe.
    
    Se filter_col o filter_val sono None, restituisce una copia del df originale.
    
    Args:
        df: dataframe da filtrare
        filter_col: nome colonna su cui filtrare
        filter_val: valore da selezionare
    
    Returns:
        Dataframe filtrato (copia)
    """
    if filter_col is None or filter_val is None:
        return df.copy()
    return df[df[filter_col] == filter_val].copy()

### 3. Analisi Cardinalità dei Match

Questa funzione analizza la "forma" dei match trovati:
- **1:1**: una visita in df1 corrisponde a una visita in df2 (caso ideale)
- **1:N**: una visita in df1 corrisponde a N visite in df2 (problema in df2)
- **N:1**: N visite in df1 corrispondono a una visita in df2 (problema in df1, tipico di VOLUMES)
- **N:N**: multiple visite in entrambi (caso raro, richiede analisi manuale)

In [ ]:
def check_match_cardinality(matches, match_type="", verbose=True):
    """
    Analizza la cardinalità dei match tra due dataset.
    
    Per ogni match, conta quante righe di df1 e df2 sono coinvolte.
    Questo aiuta a identificare duplicati o anomalie nei dati.
    
    Args:
        matches: dizionario {key: [lista_idx_df1, lista_idx_df2]}
        match_type: etichetta per il print (es. "EXACT MATCHES")
        verbose: se True, stampa i risultati
    
    Returns:
        stats: dizionario con conteggi per tipo {'1:1': n, '1:N': n, ...}
        problematic: dizionario con liste di match problematici per tipo
    """
    # Inizializza contatori
    stats = {'1:1': 0, '1:N': 0, 'N:1': 0, 'N:N': 0}
    
    # Liste per salvare i match problematici (da gestire)
    problematic = {'1:N': [], 'N:1': [], 'N:N': []}
    
    # Analizza ogni match
    for key, (idx1, idx2) in matches.items():
        n1, n2 = len(idx1), len(idx2)
        
        if n1 == 1 and n2 == 1:
            # Caso ideale: 1 a 1
            stats['1:1'] += 1
            
        elif n1 == 1 and n2 > 1:
            # 1 visita in df1 matcha N visite in df2
            stats['1:N'] += 1
            problematic['1:N'].append((key, idx1, idx2))
            
        elif n1 > 1 and n2 == 1:
            # N visite in df1 matchano 1 visita in df2
            # Tipico di VOLUMES: stesso paziente, stessa data, acquisizioni multiple
            stats['N:1'] += 1
            problematic['N:1'].append((key, idx1, idx2))
            
        else:
            # Caso complesso: multiple visite da entrambe le parti
            stats['N:N'] += 1
            problematic['N:N'].append((key, idx1, idx2))
    
    # Stampa riepilogo
    if verbose:
        print(f"\n=== {match_type} ===")
        print(f"1:1  (ok)       -> {stats['1:1']}")
        print(f"1:N  (df2 mult) -> {stats['1:N']}")
        print(f"N:1  (df1 mult) -> {stats['N:1']}")
        print(f"N:N  (entrambi) -> {stats['N:N']}")
    
    return stats, problematic

### 4. Criteri di Selezione della Riga Migliore

Quando ci sono duplicati (stesso RID + EXAMDATE), bisogna sceglierne uno.

**Ordine dei criteri (dal più al meno importante):**
1. **Meno NaN** nelle colonne volumetriche → dati più completi
2. **STATUS = 'complete'** > 'partial' > nan → completezza dichiarata
3. **FLDSTRENG maggiore** → migliore qualità acquisizione MRI (es. 3T > 1.5T)
4. **Default**: prima riga

La logica è: i dati reali (NaN) sono più importanti dei metadati dichiarati (STATUS).

In [ ]:
def count_nan_in_volume_cols(row):
    """
    Conta quanti NaN ci sono nelle colonne volumetriche di una riga.
    
    Usato come primo criterio di qualità: meno NaN = dati più completi.
    
    Args:
        row: riga del dataframe (Series)
    
    Returns:
        Numero di NaN nelle colonne volumetriche
    """
    # Considera solo le colonne volumetriche presenti nel dataframe
    cols = [c for c in VOLUME_COLS if c in row.index]
    return row[cols].isna().sum()


def compare_rows_for_selection(row1, row2):
    """
    Confronta due righe e restituisce l'indice della migliore.
    
    Applica i criteri in ordine fino a trovare un vincitore.
    
    Args:
        row1, row2: righe da confrontare (Series con .name = indice)
    
    Returns:
        Indice della riga migliore (row1.name o row2.name)
    """
    # -------------------------------------------------------------------------
    # CRITERIO 1: Meno NaN nelle colonne volumetriche
    # -------------------------------------------------------------------------
    nan1 = count_nan_in_volume_cols(row1)
    nan2 = count_nan_in_volume_cols(row2)
    
    if nan1 < nan2:
        return row1.name  # row1 ha meno NaN
    if nan2 < nan1:
        return row2.name  # row2 ha meno NaN
    
    # Parità NaN -> passa al criterio successivo
    
    # -------------------------------------------------------------------------
    # CRITERIO 2: STATUS (complete > partial > nan)
    # -------------------------------------------------------------------------
    status_priority = {'complete': 0, 'partial': 1}  # più basso = meglio
    
    s1 = row1.get('STATUS', np.nan)
    s2 = row2.get('STATUS', np.nan)
    
    # nan o valori sconosciuti hanno priorità 2 (peggiore)
    p1 = status_priority.get(s1, 2)
    p2 = status_priority.get(s2, 2)
    
    if p1 < p2:
        return row1.name  # row1 ha STATUS migliore
    if p2 < p1:
        return row2.name  # row2 ha STATUS migliore
    
    # Parità STATUS -> passa al criterio successivo
    
    # -------------------------------------------------------------------------
    # CRITERIO 3: FLDSTRENG maggiore (3T > 1.5T)
    # -------------------------------------------------------------------------
    # FLDSTRENG indica la forza del campo magnetico MRI
    f1 = pd.to_numeric(row1.get('FLDSTRENG', np.nan), errors='coerce')
    f2 = pd.to_numeric(row2.get('FLDSTRENG', np.nan), errors='coerce')
    
    if pd.notna(f1) and pd.notna(f2):
        if f1 > f2:
            return row1.name
        if f2 > f1:
            return row2.name
    elif pd.notna(f1):
        # Solo row1 ha FLDSTRENG valido
        return row1.name
    elif pd.notna(f2):
        # Solo row2 ha FLDSTRENG valido
        return row2.name
    
    # -------------------------------------------------------------------------
    # CRITERIO 4: Default - tieni la prima
    # -------------------------------------------------------------------------
    return row1.name


def select_best_from_group(df_group):
    """
    Data un gruppo di righe duplicate, seleziona la migliore.
    
    Usa compare_rows_for_selection iterativamente per trovare il vincitore.
    
    Args:
        df_group: DataFrame con le righe duplicate
    
    Returns:
        Indice della riga migliore
    """
    # Se c'è solo una riga, è automaticamente la migliore
    if len(df_group) == 1:
        return df_group.index[0]
    
    # Confronta a coppie, tenendo traccia del vincitore corrente
    best_idx = df_group.index[0]
    for idx in df_group.index[1:]:
        best_idx = compare_rows_for_selection(
            df_group.loc[best_idx], 
            df_group.loc[idx]
        )
    
    return best_idx

### 5. Rimozione Duplicati (specifico per VOLUMES)

Il dataset VOLUMES può contenere duplicati: stesso paziente (RID) con stessa data (EXAMDATE) ma acquisizioni multiple (es. ripetizioni nella stessa sessione MRI).

Questa funzione:
1. Identifica i gruppi di righe duplicate
2. Per ogni gruppo, seleziona la riga migliore usando i criteri definiti
3. Elimina le righe non selezionate

In [ ]:
def remove_duplicates_volumes(df, verbose=True, debug=False, debug_n=3):
    """
    Rimuove duplicati (stesso RID + EXAMDATE) dal dataset VOLUMES.
    
    Per ogni gruppo di duplicati, mantiene solo la riga "migliore"
    secondo i criteri di select_best_from_group.
    
    Args:
        df: dataframe VOLUMES
        verbose: se True, stampa statistiche
        debug: se True, mostra dettagli di ogni confronto
        debug_n: numero massimo di gruppi da mostrare in debug
    
    Returns:
        Dataframe senza duplicati
    """
    df = df.copy()
    
    # -------------------------------------------------------------------------
    # Trova righe duplicate (stesso RID + EXAMDATE)
    # -------------------------------------------------------------------------
    # filter() restituisce solo i gruppi con più di una riga
    duplicates = df.groupby([RID_COL, DATE_COL]).filter(lambda x: len(x) > 1)
    
    if duplicates.empty:
        if verbose:
            print("Nessun duplicato trovato")
        return df
    
    # Statistiche
    n_duplicates = len(duplicates)
    n_groups = duplicates.groupby([RID_COL, DATE_COL]).ngroups
    
    if verbose:
        print(f"Trovati {n_duplicates} righe duplicate in {n_groups} gruppi")
    
    # -------------------------------------------------------------------------
    # Per ogni gruppo, seleziona la riga migliore
    # -------------------------------------------------------------------------
    indices_to_drop = []
    debug_count = 0
    
    for (rid, date), group in duplicates.groupby([RID_COL, DATE_COL]):
        # Trova l'indice della riga migliore
        best_idx = select_best_from_group(group)
        
        # Marca per eliminazione tutte le altre righe del gruppo
        dropped = [i for i in group.index if i != best_idx]
        indices_to_drop.extend(dropped)
        
        # ---------------------------------------------------------------------
        # Debug: mostra dettagli del confronto
        # ---------------------------------------------------------------------
        if debug and debug_count < debug_n:
            print(f"\n{'='*60}")
            print(f"GRUPPO {debug_count + 1}: RID={rid}, DATE={date}")
            print(f"{'='*60}")
            
            for idx in group.index:
                row = group.loc[idx]
                nan_count = count_nan_in_volume_cols(row)
                marker = " ← MANTENUTA" if idx == best_idx else " ← ELIMINATA"
                
                print(f"\n  [{idx}]{marker}")
                print(f"      NaN count: {nan_count}")
                print(f"      STATUS: {row.get('STATUS', 'N/A')}")
                print(f"      FLDSTRENG: {row.get('FLDSTRENG', 'N/A')}")
                
                # Mostra valori volume (solo non-NaN)
                vol_vals = {c: round(row[c], 6) for c in VOLUME_COLS 
                           if c in row.index and pd.notna(row[c])}
                if vol_vals:
                    print(f"      Volumi: {vol_vals}")
            
            debug_count += 1
            
            if debug_count == debug_n and n_groups > debug_n:
                print(f"\n... altri {n_groups - debug_n} gruppi non mostrati ...")
    
    # -------------------------------------------------------------------------
    # Elimina le righe non selezionate
    # -------------------------------------------------------------------------
    df_clean = df.drop(indices_to_drop)
    
    if verbose:
        print(f"\nRimosse {len(indices_to_drop)} righe, rimaste {len(df_clean)}")
    
    return df_clean

### 6. Riduzione Match N:1 (o 1:N) a 1:1

Dopo aver trovato i match, alcuni potrebbero essere N:1 (tipico di VOLUMES) o 1:N.

Questa funzione:
- Prende i match problematici
- Per il lato con multiple righe, seleziona la migliore
- Aggiorna il dizionario matches con la versione 1:1

In [ ]:
def reduce_matches_to_one_to_one(df, matches, problematic, side='df1'):
    """
    Riduce i match N:1 o 1:N a 1:1 selezionando la riga migliore.
    
    Args:
        df: dataframe da cui selezionare (df_base per N:1, df_add per 1:N)
        matches: dizionario originale dei match
        problematic: dizionario con liste di match problematici
        side: 'df1' per ridurre N:1, 'df2' per ridurre 1:N
    
    Returns:
        matches aggiornato con tutti i match ridotti a 1:1
    """
    reduced_matches = matches.copy()
    
    # Seleziona la lista corretta di problematici
    key_list = 'N:1' if side == 'df1' else '1:N'
    
    for key, idx1, idx2 in problematic[key_list]:
        if side == 'df1':
            # N:1 -> multiple righe in df1, una in df2
            # Seleziona la migliore tra idx1
            group = df.loc[idx1]
            best_idx = select_best_from_group(group)
            reduced_matches[key] = [[best_idx], idx2]  # ora è 1:1
        else:
            # 1:N -> una riga in df1, multiple in df2
            # Seleziona la migliore tra idx2
            group = df.loc[idx2]
            best_idx = select_best_from_group(group)
            reduced_matches[key] = [idx1, [best_idx]]  # ora è 1:1
    
    print(f"Ridotti {len(problematic[key_list])} match {key_list} a 1:1")
    return reduced_matches

### 7. Filtro Buffer Matches da Mergiare

Non tutti i buffer matches (visite vicine ma non identiche) devono essere mergiati.

**Criteri per decidere se mergiare:**
- **Stesso VISCODE**: stessa visita programmata (es. entrambi "m06") → merge
- **Date < 30 giorni**: molto vicine → merge
- **VISCODE diversi con 'm'** (es. "m06" vs "m12"): visite diverse → controllo manuale
- **Altrimenti**: skip

Questo evita di mergiare erroneamente visite che sono realmente diverse.

In [ ]:
def filter_buffer_matches_to_merge(df_base, df_add, buffer_matches):
    """
    Filtra i buffer matches decidendo quali mergiare.
    
    Applica la logica:
    - Stesso VISCODE o date < 30gg → merge
    - VISCODE diversi ma entrambi con 'm' → controllo manuale
    - Altrimenti → skip
    
    Args:
        df_base: primo dataframe
        df_add: secondo dataframe
        buffer_matches: dizionario dei buffer match
    
    Returns:
        to_merge: match da mergiare
        to_skip: match da saltare
        manual_check: match che richiedono controllo manuale
    """
    to_merge = {}
    to_skip = {}
    manual_check = []
    
    for key, (idx1, idx2) in buffer_matches.items():
        # Prendi il primo indice (dopo riduzione N:1, ce n'è solo uno)
        x1 = idx1[0] if isinstance(idx1, list) else idx1
        x2 = idx2[0] if isinstance(idx2, list) else idx2
        
        # Estrai VISCODE (codice visita programmata, es. "bl", "m06", "m12")
        viscode1 = df_base.loc[x1, 'VISCODE'] if 'VISCODE' in df_base.columns else None
        viscode2 = df_add.loc[x2, 'VISCODE'] if 'VISCODE' in df_add.columns else None
        
        # Estrai date
        date1 = pd.to_datetime(df_base.loc[x1, DATE_COL])
        date2 = pd.to_datetime(df_add.loc[x2, DATE_COL])
        
        # ---------------------------------------------------------------------
        # CASO 1: VISCODE diversi ma entrambi contengono 'm' (month)
        # Es: "m06" vs "m12" → probabilmente visite diverse → controllo manuale
        # ---------------------------------------------------------------------
        if viscode1 != viscode2:
            if viscode1 and viscode2 and 'm' in str(viscode1) and 'm' in str(viscode2):
                manual_check.append((key, idx1, idx2))
                to_skip[key] = [idx1, idx2]
                continue
        
        # ---------------------------------------------------------------------
        # CASO 2: Stesso VISCODE o date molto vicine (<30gg) → merge
        # ---------------------------------------------------------------------
        if viscode1 == viscode2 or abs(date1 - date2) < pd.Timedelta(days=30):
            to_merge[key] = [idx1, idx2]
        else:
            # ---------------------------------------------------------------------
            # CASO 3: Altrimenti → skip (visite troppo diverse)
            # ---------------------------------------------------------------------
            to_skip[key] = [idx1, idx2]
    
    print(f"Buffer da mergiare: {len(to_merge)}")
    print(f"Buffer da saltare: {len(to_skip)}")
    print(f"Buffer da controllare manualmente: {len(manual_check)}")
    
    return to_merge, to_skip, manual_check

### 8. Allineamento Date per Buffer Matches

Per i buffer matches da mergiare, le date sono vicine ma non identiche.
Per permettere il merge, allineiamo le date.

**Logica:**
- Se stiamo aggiungendo **ADNIMERGE** → usa le sue date come riferimento (è il dataset "master")
- Altrimenti → usa le date di df_base come riferimento

In [ ]:
def align_dates_for_buffer(df_base, df_add, buffer_to_merge, df_add_name):
    """
    Allinea le date dei buffer matches per permettere il merge.
    
    Modifica la data di una delle due visite per farle coincidere.
    La scelta di quale modificare dipende da quale dataset è "master".
    
    Args:
        df_base: primo dataframe (verrà modificato se df_add è ADNIMERGE)
        df_add: secondo dataframe (verrà modificato se non è ADNIMERGE)
        buffer_to_merge: dizionario dei buffer match da mergiare
        df_add_name: nome del file df_add (per capire se è ADNIMERGE)
    
    Returns:
        df_base, df_add: dataframe con date allineate
    """
    df_base = df_base.copy()
    df_add = df_add.copy()
    
    for key, (idx1, idx2) in buffer_to_merge.items():
        # Estrai indici singoli
        x1 = idx1[0] if isinstance(idx1, list) else idx1
        x2 = idx2[0] if isinstance(idx2, list) else idx2
        
        df_add.loc[x2, DATE_COL] = df_base.loc[x1, DATE_COL]
    
    print(f'Date allineate: {len(buffer_to_merge)}')
    return df_base, df_add

### 9. Merge Effettivo dei Dataset

Esegue il merge vero e proprio:

1. **Visite matchate (exact + buffer)**: 
   - Copia i valori delle colonne non comuni da df_add a df_base

2. **Visite non matchate di df_add**:
   - Pazienti/visite che esistono solo in df_add
   - Vengono impilate (concatenate) a df_base

Il risultato è un dataframe che contiene tutti i dati di entrambi.

In [ ]:
def merge_datasets(df_base, df_add, exact_matches, buffer_to_merge):
    """
    Esegue il merge effettivo tra due dataset.
    
    Per le visite matchate: estende df_base con le colonne non comuni di df_add.
    Per le visite non matchate: le aggiunge in fondo a df_base.
    
    Args:
        df_base: dataframe base (verrà esteso)
        df_add: dataframe da aggiungere
        exact_matches: dizionario match esatti
        buffer_to_merge: dizionario buffer match da mergiare
    
    Returns:
        Dataframe mergiato
    """
    df_base = df_base.copy()
    df_add = df_add.copy()
    
    # -------------------------------------------------------------------------
    # Identifica colonne da aggiungere
    # -------------------------------------------------------------------------
    cols_only_add = list(df_add.columns.difference(df_base.columns))
    
    # -------------------------------------------------------------------------
    # Raccogli tutti gli indici matchati
    # -------------------------------------------------------------------------
    all_matches = {**exact_matches, **buffer_to_merge}
    matched_idx2 = set()  # indici di df_add che hanno un match
    
    for key, (idx1, idx2) in all_matches.items():
        if isinstance(idx2, list):
            matched_idx2.update(idx2)
        else:
            matched_idx2.add(idx2)
    
    # -------------------------------------------------------------------------
    # Aggiungi colonne mancanti a df_base (inizializzate a NaN)
    # -------------------------------------------------------------------------
    for col in cols_only_add:
        df_base[col] = np.nan
    
    # -------------------------------------------------------------------------
    # Per ogni match: copia i valori delle colonne non comuni
    # -------------------------------------------------------------------------
    for key, (idx1, idx2) in all_matches.items():
        # Estrai indici singoli (dopo riduzione a 1:1)
        i1 = idx1[0] if isinstance(idx1, list) else idx1
        i2 = idx2[0] if isinstance(idx2, list) else idx2
        
        # Copia i valori
        for col in cols_only_add:
            df_base.loc[i1, col] = df_add.loc[i2, col]
    
    # -------------------------------------------------------------------------
    # Identifica righe di df_add non matchate (da aggiungere)
    # -------------------------------------------------------------------------
    unmatched_idx2 = df_add.index.difference(matched_idx2)
    df_add_unmatched = df_add.loc[unmatched_idx2].copy()
    
    # Assicura che abbia le stesse colonne di df_base
    for col in df_base.columns:
        if col not in df_add_unmatched.columns:
            df_add_unmatched[col] = np.nan
    
    # Riordina colonne per matchare df_base
    df_add_unmatched = df_add_unmatched[df_base.columns]
    
    # -------------------------------------------------------------------------
    # Concatena
    # -------------------------------------------------------------------------
    result = pd.concat([df_base, df_add_unmatched], ignore_index=True)
    
    print(f"\n--- Risultato merge ---")
    print(f"Righe matchate: {len(all_matches)}")
    print(f"Righe aggiunte da df_add: {len(df_add_unmatched)}")
    print(f"Totale finale: {len(result)}")
    
    return result

## Start

In [ ]:
# =============================================================================
# SELEZIONE DATASET
# =============================================================================

# Indici dei dataset da mergiare
# Modifica questi valori per cambiare i dataset
i = 0  # Indice df_base (primo dataset)
j = 1  # Indice df_add (secondo dataset)

# Nomi file
df1_name = keys[i]
df2_name = keys[j]

print(f"Dataset selezionati:")
print(f"  df_base (i={i}): {df1_name}")
print(f"  df_add  (j={j}): {df2_name}")

Ogni dataset può avere diverse versioni/metodi.
Questa cella genera tutte le combinazioni possibili tra i due dataset.

Es: se df1 ha FSVERSION [5.1, 6.0] e df2 non ha filtro:
- Combinazione 1: FSVERSION=5.1 × None
- Combinazione 2: FSVERSION=6.0 × None

In [ ]:
df_1 = dfs[df1_name].copy()
df_2 = dfs[df2_name].copy()

combinations = get_filter_combinations(df_1, df_2, df1_name, df2_name)
print(f"\nCombinazioni da processare: {len(combinations)}")
for c in combinations:
    print(f"  {c['df1_filter_val']} x {c['df2_filter_val']}")

In [ ]:
# =============================================================================
# SELEZIONE COMBINAZIONE
# =============================================================================

# Indice della combinazione da processare
# Modifica questo valore per cambiare combinazione
comb_idx = 0

# Estrai la combinazione
comb = combinations[comb_idx]

# Mostra dettagli
print(f"Combinazione selezionata: [{comb_idx}]")
print(f"  df_base filtro: {comb['df1_filter_col']} = {comb['df1_filter_val']}")
print(f"  df_add  filtro: {comb['df2_filter_col']} = {comb['df2_filter_val']}")

In [ ]:
# =============================================================================
# APPLICA FILTRI
# =============================================================================

df_base = apply_filter(df_1, comb['df1_filter_col'], comb['df1_filter_val'])
df_add = apply_filter(df_2, comb['df2_filter_col'], comb['df2_filter_val'])

print(f"Dimensioni dopo filtro:")
print(f"  df_base: {len(df_base)} righe")
print(f"  df_add:  {len(df_add)} righe")

# Controllo colonne in comune e non
cols_common = list(df_base.columns.intersection(df_add.columns))
cols_only_base = list(df_base.columns.difference(df_add.columns))
cols_only_add = list(df_add.columns.difference(df_base.columns))

print(f"\nColonne:")
print(f"  In comune: {len(cols_common)}")
print(f"  Solo df_base: {len(cols_only_base)}")
print(f"  Solo df_add: {len(cols_only_add)}")

if cols_only_base:
    print(f"\n  Dettaglio solo df_base: {cols_only_base}")
if cols_only_add:
    print(f"\n  Dettaglio solo df_add: {cols_only_add}")

Se uno dei dataset è VOLUMES, rimuove i duplicati (stesso RID + EXAMDATE).

Questa cella viene eseguita automaticamente solo se necessario.

In [ ]:
if 'VOLUMES' in df1_name.upper():
    print("--- Rimozione duplicati da df_base (VOLUMES) ---")
    df_base = remove_duplicates_volumes(df_base, debug=True, debug_n=10)

if 'VOLUMES' in df2_name.upper():
    print("\n--- Rimozione duplicati da df_add (VOLUMES) ---")
    df_add = remove_duplicates_volumes(df_add, debug=True, debug_n=10)

Trova le corrispondenze tra i due dataset:
- **Exact matches**: stesso RID e stessa EXAMDATE
- **Buffer matches**: stesso RID, date diverse ma entro BUFFER_DAYS

In [ ]:
exact_matches, buffer_matches = mergeTools.find_visit_matches(df_base, df_add, buffer_days=BUFFER_DAYS)
mergeTools.verify_visit_matches(exact_matches, buffer_matches)

exact_index1, exact_index2 = mergeTools.list_index_visit_matches(exact_matches)
buff_index1, buff_index2 = mergeTools.list_index_visit_matches(buffer_matches)

print(f"\nExact matches: {len(exact_index1)}")
print(f"Buffer matches: {len(buff_index1)}")

Verifica la "forma" dei match:
- **1:1** è il caso ideale
- **N:1** indica duplicati in df_base (tipico di VOLUMES)
- **1:N** indica duplicati in df_add
- **N:N** richiede analisi manuale

In [ ]:
# =============================================================================
# ANALISI CARDINALITÀ
# =============================================================================

print("Analisi cardinalità match...\n")

# Analizza exact matches
stats_exact, prob_exact = check_match_cardinality(exact_matches, "EXACT MATCHES")

# Analizza buffer matches
stats_buffer, prob_buffer = check_match_cardinality(buffer_matches, "BUFFER MATCHES")

# Riepilogo problemi
print("\n" + "="*60)
print("RIEPILOGO PROBLEMI")
print("="*60)

total_problems = (
    len(prob_exact['1:N']) + len(prob_exact['N:1']) + len(prob_exact['N:N']) +
    len(prob_buffer['1:N']) + len(prob_buffer['N:1']) + len(prob_buffer['N:N'])
)

if total_problems == 0:
    print("✓ Nessun problema trovato, tutti i match sono 1:1")
else:
    print(f"Trovati {total_problems} match problematici da gestire")

In [ ]:
# =============================================================================
# GESTIONE EXACT MATCHES PROBLEMATICI
# =============================================================================

# Flag per identificare problemi bloccanti
has_problems_exact = bool(prob_exact['1:N'] or prob_exact['N:N'])
has_n1_exact = bool(prob_exact['N:1'])

if has_problems_exact:
    # -----------------------------------------
    # STOP: problemi che richiedono intervento manuale
    # -----------------------------------------
    print("="*60)
    print("⚠️  STOP: Match esatti con problemi non gestibili automaticamente")
    print("="*60)
    print(f"  1:N (df_add multipli): {len(prob_exact['1:N'])}")
    print(f"  N:N (entrambi multipli): {len(prob_exact['N:N'])}")
    print("\nRichiesto controllo manuale prima di procedere.")
    print("Analizza prob_exact['1:N'] e prob_exact['N:N'] per dettagli.")
    
elif has_n1_exact:
    # -----------------------------------------
    # N:1: riducibile automaticamente
    # -----------------------------------------
    print("="*60)
    print(f"Riduzione {len(prob_exact['N:1'])} match esatti N:1 a 1:1")
    print("="*60)
    exact_matches = reduce_matches_to_one_to_one(
        df_base, 
        exact_matches, 
        prob_exact, 
        side='df1'
    )
    
else:
    print("✓ Match esatti tutti 1:1, nessuna azione necessaria")

In [ ]:
# =============================================================================
# GESTIONE BUFFER MATCHES PROBLEMATICI
# =============================================================================

has_problems_buffer = bool(prob_buffer['1:N'] or prob_buffer['N:N'])
has_n1_buffer = bool(prob_buffer['N:1'])

if has_problems_buffer:
    # -----------------------------------------
    # STOP: problemi che richiedono intervento manuale
    # -----------------------------------------
    print("="*60)
    print("⚠️  STOP: Buffer matches con problemi non gestibili automaticamente")
    print("="*60)
    print(f"  1:N (df_add multipli): {len(prob_buffer['1:N'])}")
    print(f"  N:N (entrambi multipli): {len(prob_buffer['N:N'])}")
    print("\nRichiesto controllo manuale prima di procedere.")
    print("Analizza prob_buffer['1:N'] e prob_buffer['N:N'] per dettagli.")
    
elif has_n1_buffer:
    # -----------------------------------------
    # N:1: riducibile automaticamente
    # -----------------------------------------
    print("="*60)
    print(f"Riduzione {len(prob_buffer['N:1'])} buffer N:1 a 1:1")
    print("="*60)
    buffer_matches = reduce_matches_to_one_to_one(
        df_base, 
        buffer_matches, 
        prob_buffer, 
        side='df1'
    )
    
else:
    print("✓ Buffer matches tutti 1:1, nessuna azione necessaria")

Decide quali buffer matches devono essere effettivamente mergiati.

I buffer con VISCODE incompatibili vengono saltati o segnalati per controllo manuale.

In [ ]:
buffer_to_merge, buffer_to_skip, manual_check = filter_buffer_matches_to_merge(
    df_base, df_add, buffer_matches
)

if manual_check:
    print(f"\n⚠️  Attenzione: {len(manual_check)} buffer richiedono controllo manuale")
    for key, idx1, idx2 in manual_check[:5]:  # mostra primi 5
        print(f"  {key}")

Ultimo controllo prima del merge: verifica che i buffer selezionati siano tutti 1:1.

In [ ]:
if buffer_to_merge:
    stats_merge, prob_merge = check_match_cardinality(buffer_to_merge, "BUFFER TO MERGE")
    
    if prob_merge['N:1']:
        print(f"\nRiduzione {len(prob_merge['N:1'])} buffer-to-merge N:1 a 1:1...")
        buffer_to_merge = reduce_matches_to_one_to_one(df_base, buffer_to_merge, prob_merge, side='df1')
else:
    print("Nessun buffer match da mergiare")

Allinea le date dei buffer matches per permettere il merge.

In [ ]:
if buffer_to_merge:
    df_base, df_add = align_dates_for_buffer(df_base, df_add, buffer_to_merge, df2_name)
else:
    print("Nessun allineamento date necessario")

### Merge Finale

In [ ]:
df_merged = merge_datasets(df_base, df_add, exact_matches, buffer_to_merge)

In [ ]:
def analyze_merged_dataset(df, df_name="Dataset"):
    """
    Statistiche essenziali su un dataset mergiato.
    
    Args:
        df: dataframe da analizzare
        df_name: nome per i print
    """
    print(f"\n{'='*50}")
    print(f"  STATISTICHE: {df_name}")
    print(f"{'='*50}")
    
    # Dimensioni
    print(f"\nDimensioni: {len(df):,} righe × {len(df.columns)} colonne")
    
    # Soggetti e visite
    n_soggetti = df[RID_COL].nunique()
    visite_per_soggetto = df.groupby(RID_COL).size()
    
    print(f"\nSoggetti: {n_soggetti:,}")
    print(f"Visite per soggetto:")
    print(f"  Min: {visite_per_soggetto.min()}")
    print(f"  Max: {visite_per_soggetto.max()}")
    print(f"  Media: {visite_per_soggetto.mean():.2f}")
    print(f"  Mediana: {visite_per_soggetto.median():.1f}")
    print(f"  Std: {visite_per_soggetto.std():.2f}")
    
    # Distribuzione visite (quartili)
    print(f"  Quartili: {visite_per_soggetto.quantile([0.25, 0.5, 0.75]).to_dict()}")
    
    # Range temporale
    if DATE_COL in df.columns:
        date_min = pd.to_datetime(df[DATE_COL]).min()
        date_max = pd.to_datetime(df[DATE_COL]).max()
        print(f"\nRange date: {date_min.strftime('%Y-%m-%d')} → {date_max.strftime('%Y-%m-%d')}")
    
    # NaN tutte le colonne
    nan_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
    
    print(f"\nNaN per colonna:")
    for col, pct in nan_pct.items():
        print(f"  {col}: {pct:.1f}%")

In [ ]:
analyze_merged_dataset(df_merged, "Merge VOLUMES + SCALES")

In [ ]:
display(df_merged)

In [ ]:
# Salva il risultato per la prossima iterazione
# comb_key = f"{comb['df1_filter_val']}_{comb['df2_filter_val']}"
# results = {comb_key: df_merged}

# print(f"\nMerge completato: {comb_key}")
# print(f"Righe: {len(df_merged)}")
# print(f"Colonne: {len(df_merged.columns)}")

In [ ]:
# Per il prossimo step, usa df_merged come df_base
# Esempio:
# df_1 = df_merged.copy()
# df_2 = dfs[keys[2]].copy()
# df1_name = 'merged_step1'
# df2_name = keys[2]
# 
# Poi riesegui dalla CELLA 12 in poi

## TEST

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_test_datasets(seed=42):
    """
    Genera dataset di test con tutti i casi particolari.
    
    Returns:
        df_volumes: simula VOLUMES con duplicati
        df_scales: simula SCALES senza duplicati
        df_csf: simula CSF con metodi diversi
    """
    np.random.seed(seed)
    
    # =========================================================================
    # BASE DATES E RIDS
    # =========================================================================
    base_date = datetime(2020, 1, 15)
    
    # RID comuni a tutti i dataset
    common_rids = [1001, 1002, 1003, 1004, 1005]
    
    # RID solo in VOLUMES
    only_vol_rids = [2001, 2002]
    
    # RID solo in SCALES
    only_scale_rids = [3001, 3002]
    
    # =========================================================================
    # DATASET 1: VOLUMES (con duplicati e versioni multiple)
    # =========================================================================
    vol_rows = []
    
    # --- Caso 1: Duplicati (stesso RID, stessa data, acquisizioni multiple) ---
    # RID 1001: 2 acquisizioni stessa data, una completa, una parziale
    vol_rows.append({
        'RID': 1001, 'EXAMDATE': base_date, 'VISCODE': 'bl',
        'FSVERSION': '6.0', 'STATUS': 'complete', 'FLDSTRENG': 3.0,
        'Ventricles%ICV': 0.02, 'Hippocampus%ICV': 0.005,
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    vol_rows.append({
        'RID': 1001, 'EXAMDATE': base_date, 'VISCODE': 'bl',
        'FSVERSION': '6.0', 'STATUS': 'partial', 'FLDSTRENG': 3.0,
        'Ventricles%ICV': 0.02, 'Hippocampus%ICV': np.nan,  # NaN!
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': np.nan,    # NaN!
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    
    # RID 1002: 3 acquisizioni stessa data (caso N:1 con N=3)
    for i in range(3):
        nan_count = i  # 0, 1, 2 NaN progressivi
        vol_rows.append({
            'RID': 1002, 'EXAMDATE': base_date, 'VISCODE': 'bl',
            'FSVERSION': '6.0', 'STATUS': 'complete' if i == 0 else 'partial',
            'FLDSTRENG': 3.0 - i * 0.5,
            'Ventricles%ICV': 0.02 if i < 1 else np.nan,
            'Hippocampus%ICV': 0.005 if i < 2 else np.nan,
            'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
            'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
        })
    
    # --- Caso 2: Match esatti normali (1:1) ---
    # RID 1003: visita singola, match esatto atteso
    vol_rows.append({
        'RID': 1003, 'EXAMDATE': base_date + timedelta(days=30), 'VISCODE': 'm01',
        'FSVERSION': '6.0', 'STATUS': 'complete', 'FLDSTRENG': 3.0,
        'Ventricles%ICV': 0.025, 'Hippocampus%ICV': 0.004,
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    
    # --- Caso 3: Buffer match (date vicine ma non identiche) ---
    # RID 1004: data leggermente diversa da SCALES (entro buffer)
    vol_rows.append({
        'RID': 1004, 'EXAMDATE': base_date + timedelta(days=60), 'VISCODE': 'm02',
        'FSVERSION': '6.0', 'STATUS': 'complete', 'FLDSTRENG': 1.5,
        'Ventricles%ICV': 0.03, 'Hippocampus%ICV': 0.004,
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    
    # --- Caso 4: Buffer match con VISCODE diversi (controllo manuale) ---
    # RID 1005: VISCODE m06 vs m12 in SCALES
    vol_rows.append({
        'RID': 1005, 'EXAMDATE': base_date + timedelta(days=180), 'VISCODE': 'm06',
        'FSVERSION': '6.0', 'STATUS': 'complete', 'FLDSTRENG': 3.0,
        'Ventricles%ICV': 0.028, 'Hippocampus%ICV': 0.0045,
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    
    # --- Caso 5: Nessun match (solo in VOLUMES) ---
    for rid in only_vol_rids:
        vol_rows.append({
            'RID': rid, 'EXAMDATE': base_date + timedelta(days=90), 'VISCODE': 'm03',
            'FSVERSION': '7.0', 'STATUS': 'complete', 'FLDSTRENG': 3.0,
            'Ventricles%ICV': 0.022, 'Hippocampus%ICV': 0.005,
            'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
            'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
        })
    
    # --- Caso 6: Versione diversa (FSVERSION 5.1) ---
    vol_rows.append({
        'RID': 1003, 'EXAMDATE': base_date + timedelta(days=30), 'VISCODE': 'm01',
        'FSVERSION': '5.1', 'STATUS': 'complete', 'FLDSTRENG': 1.5,
        'Ventricles%ICV': 0.024, 'Hippocampus%ICV': 0.0042,
        'Entorhinal%ICV': 0.001, 'Fusiform%ICV': 0.01,
        'MidTemp%ICV': 0.02, 'ICV%ICV': 1.0
    })
    
    df_volumes = pd.DataFrame(vol_rows)
    df_volumes['EXAMDATE'] = pd.to_datetime(df_volumes['EXAMDATE'])
    df_volumes['FSVERSION'] = df_volumes['FSVERSION'].astype(str)
    
    # =========================================================================
    # DATASET 2: SCALES (senza duplicati, senza filtro versione)
    # =========================================================================
    scale_rows = []
    
    # RID 1001: match esatto con VOLUMES (dopo rimozione duplicati)
    scale_rows.append({
        'RID': 1001, 'EXAMDATE': base_date, 'VISCODE': 'bl',
        'MMSE': 28, 'CDRSB': 0.5, 'ADAS13': 12, 'FAQ': 2
    })
    
    # RID 1002: match esatto (dopo rimozione duplicati da VOLUMES)
    scale_rows.append({
        'RID': 1002, 'EXAMDATE': base_date, 'VISCODE': 'bl',
        'MMSE': 26, 'CDRSB': 1.0, 'ADAS13': 15, 'FAQ': 5
    })
    
    # RID 1003: match esatto
    scale_rows.append({
        'RID': 1003, 'EXAMDATE': base_date + timedelta(days=30), 'VISCODE': 'm01',
        'MMSE': 27, 'CDRSB': 0.5, 'ADAS13': 10, 'FAQ': 1
    })
    
    # RID 1004: buffer match (data +15 giorni rispetto a VOLUMES)
    scale_rows.append({
        'RID': 1004, 'EXAMDATE': base_date + timedelta(days=75), 'VISCODE': 'm02',
        'MMSE': 25, 'CDRSB': 1.5, 'ADAS13': 18, 'FAQ': 8
    })
    
    # RID 1005: buffer match con VISCODE diverso (m12 vs m06)
    scale_rows.append({
        'RID': 1005, 'EXAMDATE': base_date + timedelta(days=200), 'VISCODE': 'm12',
        'MMSE': 24, 'CDRSB': 2.0, 'ADAS13': 22, 'FAQ': 10
    })
    
    # RID solo in SCALES
    for rid in only_scale_rids:
        scale_rows.append({
            'RID': rid, 'EXAMDATE': base_date + timedelta(days=45), 'VISCODE': 'm01',
            'MMSE': 29, 'CDRSB': 0.0, 'ADAS13': 8, 'FAQ': 0
        })
    
    df_scales = pd.DataFrame(scale_rows)
    df_scales['EXAMDATE'] = pd.to_datetime(df_scales['EXAMDATE'])
    
    # =========================================================================
    # DATASET 3: CSF (con metodi diversi per testare combinazioni)
    # =========================================================================
    csf_rows = []
    
    # RID 1001, 1002: con method_CSF = 'UPLC'
    for rid in [1001, 1002]:
        csf_rows.append({
            'RID': rid, 'EXAMDATE': base_date, 'VISCODE': 'bl',
            'method_CSF': 'UPLC', 'ABETA': 980, 'TAU': 250, 'PTAU': 22
        })
    
    # RID 1003: con method_CSF = 'ELISA'
    csf_rows.append({
        'RID': 1003, 'EXAMDATE': base_date + timedelta(days=30), 'VISCODE': 'm01',
        'method_CSF': 'ELISA', 'ABETA': 850, 'TAU': 280, 'PTAU': 28
    })
    
    df_csf = pd.DataFrame(csf_rows)
    df_csf['EXAMDATE'] = pd.to_datetime(df_csf['EXAMDATE'])
    
    return df_volumes, df_scales, df_csf


# Genera i dataset
df_vol_test, df_scale_test, df_csf_test = generate_test_datasets()

print("Dataset di test generati:\n")
print(f"df_vol_test:   {len(df_vol_test)} righe  (simula VOLUMES con duplicati)")
print(f"df_scale_test: {len(df_scale_test)} righe  (simula SCALES)")
print(f"df_csf_test:   {len(df_csf_test)} righe  (simula CSF con metodi)")

In [ ]:
print("="*80)
print("VOLUMES TEST (con duplicati)")
print("="*80)
print(df_vol_test.to_string())

print("\n" + "="*80)
print("SCALES TEST")
print("="*80)
print(df_scale_test.to_string())

print("\n" + "="*80)
print("CSF TEST (con metodi)")
print("="*80)
print(df_csf_test.to_string())

In [ ]:
test_cases = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                           CASI DI TEST INCLUSI                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  DUPLICATI IN VOLUMES (N:1)                                                  ║
║  ─────────────────────────                                                   ║
║  • RID 1001: 2 righe stessa data → 1 complete, 1 partial con NaN            ║
║  • RID 1002: 3 righe stessa data → N:1 con N=3, NaN progressivi             ║
║                                                                              ║
║  MATCH ESATTI (1:1)                                                          ║
║  ─────────────────                                                           ║
║  • RID 1001: VOLUMES ↔ SCALES (dopo rimozione duplicati)                    ║
║  • RID 1002: VOLUMES ↔ SCALES (dopo rimozione duplicati)                    ║
║  • RID 1003: VOLUMES ↔ SCALES (match diretto)                               ║
║                                                                              ║
║  BUFFER MATCHES                                                              ║
║  ──────────────                                                              ║
║  • RID 1004: date a 15gg di distanza, stesso VISCODE → merge                ║
║  • RID 1005: date a 20gg, VISCODE diversi (m06 vs m12) → manual check       ║
║                                                                              ║
║  NESSUN MATCH                                                                ║
║  ────────────                                                                ║
║  • RID 2001, 2002: solo in VOLUMES → verranno mantenuti                     ║
║  • RID 3001, 3002: solo in SCALES → verranno aggiunti                       ║
║                                                                              ║
║  VERSIONI/METODI MULTIPLI                                                    ║
║  ────────────────────────                                                    ║
║  • VOLUMES: FSVERSION 6.0 e 5.1 → 2 combinazioni                            ║
║  • CSF: method_CSF UPLC e ELISA → 2 combinazioni                            ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(test_cases)

In [ ]:
print("="*80)
print("TEST: Rimozione Duplicati")
print("="*80)

# Filtra solo versione 6.0 per il test
df_vol_6 = df_vol_test[df_vol_test['FSVERSION'] == '6.0'].copy()

print(f"\nPRIMA della rimozione duplicati ({len(df_vol_6)} righe):")
print("-"*60)

# Mostra duplicati
for rid in [1001, 1002]:
    dup = df_vol_6[df_vol_6['RID'] == rid]
    print(f"\nRID {rid}: {len(dup)} righe")
    print(dup[['RID', 'EXAMDATE', 'STATUS', 'FLDSTRENG', 
               'Ventricles%ICV', 'Hippocampus%ICV']].to_string(index=True))

# Applica rimozione
df_vol_clean = remove_duplicates_volumes(df_vol_6)

print(f"\n\nDOPO la rimozione duplicati ({len(df_vol_clean)} righe):")
print("-"*60)

for rid in [1001, 1002]:
    kept = df_vol_clean[df_vol_clean['RID'] == rid]
    print(f"\nRID {rid}: {len(kept)} riga mantenuta")
    print(kept[['RID', 'EXAMDATE', 'STATUS', 'FLDSTRENG',
                'Ventricles%ICV', 'Hippocampus%ICV']].to_string(index=True))

# Verifica
print("\n" + "="*60)
print("VERIFICA:")
rid_1001_status = df_vol_clean[df_vol_clean['RID'] == 1001]['STATUS'].values[0]
rid_1002_status = df_vol_clean[df_vol_clean['RID'] == 1002]['STATUS'].values[0]

assert rid_1001_status == 'complete', f"RID 1001: atteso 'complete', ottenuto '{rid_1001_status}'"
assert rid_1002_status == 'complete', f"RID 1002: atteso 'complete', ottenuto '{rid_1002_status}'"

print("✓ RID 1001: correttamente selezionata riga 'complete'")
print("✓ RID 1002: correttamente selezionata riga 'complete'")

In [ ]:
print("="*80)
print("TEST: Cardinalità Matches (PRIMA della rimozione duplicati)")
print("="*80)

# Usa VOLUMES con duplicati vs SCALES
df_vol_6_raw = df_vol_test[df_vol_test['FSVERSION'] == '6.0'].copy()

# Trova matches SENZA rimuovere duplicati
exact_raw, buffer_raw = mergeTools.find_visit_matches(
    df_vol_6_raw, 
    df_scale_test, 
    buffer_days=BUFFER_DAYS
)

print("\nMatches PRIMA della rimozione duplicati:")
stats_raw, prob_raw = check_match_cardinality(exact_raw, "EXACT (con duplicati)")

print("\n" + "-"*60)
print("Dettaglio match N:1:")
for key, idx1, idx2 in prob_raw['N:1']:
    print(f"  {key}: {len(idx1)} righe in VOLUMES → {len(idx2)} riga in SCALES")
    print(f"    Indici VOLUMES: {idx1}")

# Verifica
assert len(prob_raw['N:1']) >= 2, "Attesi almeno 2 match N:1"
print("\n✓ Correttamente identificati match N:1")

In [ ]:
print("="*80)
print("TEST: Pipeline Completa VOLUMES + SCALES")
print("="*80)

# Setup nomi (simulano i nomi reali)
df1_name_test = 'VOLUMES_test.csv'
df2_name_test = 'SCALEMERGE_test.csv'

# Step 1: Combinazioni
print("\n--- Step 1: Combinazioni ---")
combinations_test = get_filter_combinations(
    df_vol_test, df_scale_test, 
    df1_name_test, df2_name_test
)
print(f"Combinazioni trovate: {len(combinations_test)}")
for c in combinations_test:
    print(f"  FSVERSION={c['df1_filter_val']} × None")

# Step 2: Processa prima combinazione (FSVERSION=6.0)
print("\n--- Step 2: Filtro FSVERSION=6.0 ---")
comb_test = combinations_test[0]  # 6.0
df_base_test = apply_filter(df_vol_test, comb_test['df1_filter_col'], comb_test['df1_filter_val'])
df_add_test = df_scale_test.copy()
print(f"df_base: {len(df_base_test)} righe")
print(f"df_add:  {len(df_add_test)} righe")

# Step 3: Rimozione duplicati
print("\n--- Step 3: Rimozione Duplicati ---")
df_base_test = remove_duplicates_volumes(df_base_test)

# Step 4: Trova matches
print("\n--- Step 4: Trova Matches ---")
exact_test, buffer_test = mergeTools.find_visit_matches(df_base_test, df_add_test, buffer_days=BUFFER_DAYS)
mergeTools.verify_visit_matches(exact_test, buffer_test)

print(f"Exact matches: {len(exact_test)}")
print(f"Buffer matches: {len(buffer_test)}")

# Step 5: Cardinalità
print("\n--- Step 5: Cardinalità ---")
stats_ex, prob_ex = check_match_cardinality(exact_test, "EXACT")
stats_buf, prob_buf = check_match_cardinality(buffer_test, "BUFFER")

# Step 6: Filtra buffer
print("\n--- Step 6: Filtra Buffer ---")
buf_merge, buf_skip, manual = filter_buffer_matches_to_merge(df_base_test, df_add_test, buffer_test)

# Step 7: Allinea date
print("\n--- Step 7: Allinea Date ---")
if buf_merge:
    df_base_test, df_add_test = align_dates_for_buffer(df_base_test, df_add_test, buf_merge, df2_name_test)

# Step 8: Merge
print("\n--- Step 8: Merge Finale ---")
df_merged_test = merge_datasets(df_base_test, df_add_test, exact_test, buf_merge)

# Verifica risultato
print("\n" + "="*60)
print("VERIFICA RISULTATO")
print("="*60)

print(f"\nRighe finali: {len(df_merged_test)}")
print(f"Colonne finali: {len(df_merged_test.columns)}")

# Verifica che le colonne di entrambi ci siano
vol_cols = ['Ventricles%ICV', 'Hippocampus%ICV']
scale_cols = ['MMSE', 'CDRSB']

for col in vol_cols + scale_cols:
    assert col in df_merged_test.columns, f"Manca colonna {col}"
print(f"✓ Presenti colonne VOLUMES: {vol_cols}")
print(f"✓ Presenti colonne SCALES: {scale_cols}")

# Verifica RID
all_rids = set(df_merged_test['RID'])
expected_rids = {1001, 1002, 1003, 1004, 1005, 2001, 2002, 3001, 3002}
assert all_rids == expected_rids, f"RID mancanti o extra: {expected_rids.symmetric_difference(all_rids)}"
print(f"✓ Tutti i RID presenti: {sorted(all_rids)}")

# Verifica merge effettivo (RID con dati da entrambi)
rid_1001 = df_merged_test[df_merged_test['RID'] == 1001].iloc[0]
assert pd.notna(rid_1001['Ventricles%ICV']), "RID 1001: manca dato VOLUMES"
assert pd.notna(rid_1001['MMSE']), "RID 1001: manca dato SCALES"
print(f"✓ RID 1001: dati mergiati correttamente")

print("\n" + "="*60)
print("✅ TUTTI I TEST PASSATI!")
print("="*60)

Verifica che il buffer match RID 1005 (m06 vs m12) sia correttamente flaggato per controllo manuale.

In [ ]:
print("="*80)
print("TEST: Caso Controllo Manuale (VISCODE diversi)")
print("="*80)

# Il RID 1005 ha VISCODE m06 in VOLUMES e m12 in SCALES
print("\nDati RID 1005:")
print("  VOLUMES:", df_vol_test[df_vol_test['RID'] == 1005][['RID', 'EXAMDATE', 'VISCODE']].to_string(index=False))
print("  SCALES: ", df_scale_test[df_scale_test['RID'] == 1005][['RID', 'EXAMDATE', 'VISCODE']].to_string(index=False))

# Verifica che sia nei manual_check
if manual:
    manual_rids = [key[0] for key, _, _ in manual]
    if 1005 in manual_rids:
        print("\n✓ RID 1005 correttamente flaggato per controllo manuale")
        print(f"  Motivo: VISCODE diversi entrambi con 'm' (m06 vs m12)")
    else:
        print("\n⚠️ RID 1005 NON trovato nei manual_check")
else:
    print("\n⚠️ Nessun manual_check trovato")

Verifica che vengano generate le combinazioni corrette per dataset con metodi multipli.

In [ ]:
print("="*80)
print("TEST: Combinazioni CSF (metodi multipli)")
print("="*80)

df1_name_csf = 'VOLUMES_test.csv'
df2_name_csf = 'CSFMERGE_test.csv'

comb_csf = get_filter_combinations(df_vol_test, df_csf_test, df1_name_csf, df2_name_csf)

print(f"\nCombinazioni VOLUMES × CSF: {len(comb_csf)}")
print("-"*40)

for i, c in enumerate(comb_csf):
    v1 = c['df1_filter_val'] or 'All'
    v2 = c['df2_filter_val'] or 'All'
    print(f"  [{i}] FSVERSION={v1} × method_CSF={v2}")

# Verifica
expected_combs = 3 * 2  # 2 FSVERSION × 2 method_CSF
assert len(comb_csf) == expected_combs, f"Attese {expected_combs} combinazioni, trovate {len(comb_csf)}"
print(f"\n✓ Correttamente generate {expected_combs} combinazioni (2 FSVERSION × 2 method_CSF)")